In [168]:
import os
import pyspark
from pyspark.sql.functions import (when, col, lit, regexp_extract, count, sum as _sum, 
count_distinct, dense_rank, lower, hour, hours, date_part, extract, date_diff)
from pyspark.sql import Window
from pyspark.sql import SparkSession

In [143]:
from dotenv import load_dotenv

In [3]:
sc = SparkSession \
    .builder \
    .appName("Lms spark task") \
    .config("spark.jars", "drivers/postgresql-42.7.8.jar") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/03 10:28:08 WARN Utils: Your hostname, TUF-FX505DV, resolves to a loopback address: 127.0.1.1; using 10.231.20.122 instead (on interface enp2s0)
25/12/03 10:28:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/12/03 10:28:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
load_dotenv()
url_connection = os.environ.get("DB_POSTGRES_CONNECTION") + os.environ.get("DB_NAME")
properties = {
    "user": os.environ.get("DB_USER"),
    "password": os.environ.get("DB_PASSWORD"),
    "driver": "org.postgresql.Driver"
}

In [5]:
url_connection, properties

('jdbc:postgresql://localhost:5432/pagila',
 {'user': 'viyachikhh',
  'password': 'Lqu157268sssS',
  'driver': 'org.postgresql.Driver'})

In [6]:
df_film = sc.read.jdbc(url_connection, "film", properties=properties)
df_category = sc.read.jdbc(url_connection, "category", properties=properties)
df_actor = sc.read.jdbc(url_connection, "actor", properties=properties)
df_film_category = sc.read.jdbc(url_connection, "film_category", properties=properties)
df_film_actor = sc.read.jdbc(url_connection, "film_actor", properties=properties)

In [19]:
df_inventory = sc.read.jdbc(url_connection, "inventory", properties=properties)
df_rental = sc.read.jdbc(url_connection, "rental", properties=properties)
df_payment = sc.read.jdbc(url_connection, "payment", properties=properties)

In [76]:
df_customer = sc.read.jdbc(url_connection, "customer", properties=properties)
df_address = sc.read.jdbc(url_connection, "address", properties=properties)
df_city = sc.read.jdbc(url_connection, "city", properties=properties)

## Task 1

In [8]:
df_joined_one = df_film_category.join(df_category, (df_category['category_id'] == df_film_category['category_id']))
df_cat_count = df_joined_one.groupby(df_category['name']).agg(count('*').alias('count')).orderBy(col('count').desc())

In [9]:
df_cat_count.show()

+-----------+-----+
|       name|count|
+-----------+-----+
|      Drama|  152|
|      Music|  152|
|     Travel|  151|
|    Foreign|  150|
|      Games|  150|
|   Children|  150|
|     Action|  149|
|     Sci-Fi|  149|
|  Animation|  148|
|     Family|  147|
|   Classics|  147|
|        New|  147|
|     Sports|  145|
|Documentary|  145|
|     Comedy|  143|
|     Horror|  142|
+-----------+-----+



## Task 2

In [13]:
df_joined_two = df_film_actor.join(df_film, (df_film_actor['film_id'] == df_film['film_id'])).\
                        join(df_inventory, (df_inventory['film_id'] == df_film['film_id'])).\
                        join(df_rental, (df_rental['inventory_id'] == df_inventory['inventory_id']))
df_top_id_actors = df_joined_two.groupBy(df_film_actor['actor_id']).agg(count('*').alias('count')).orderBy(col('count').desc()).limit(10)
df_top_actors_joined = df_top_id_actors.join(df_actor, (df_top_id_actors['actor_id'] == df_actor['actor_id']))
df_top_full_names = df_top_actors_joined.select([df_film_actor['actor_id'],"first_name", "last_name", "count"]).orderBy(col('count').desc())

In [14]:
df_top_full_names.show()

+--------+----------+-----------+-----+
|actor_id|first_name|  last_name|count|
+--------+----------+-----------+-----+
|     107|      GINA|  DEGENERES|  753|
|     181|   MATTHEW|     CARREY|  678|
|     198|      MARY|     KEITEL|  674|
|     144|    ANGELA|WITHERSPOON|  654|
|     102|    WALTER|       TORN|  640|
|      60|     HENRY|      BERRY|  612|
|     150|     JAYNE|      NOLTE|  611|
|      37|       VAL|     BOLGER|  605|
|      23|    SANDRA|     KILMER|  604|
|      90|      SEAN|    GUINESS|  599|
+--------+----------+-----------+-----+



## Task 3

In [25]:
df_joined_three = df_film_category.join(df_film, (df_film_category['film_id'] == df_film['film_id'])).\
                        join(df_inventory, (df_inventory['film_id'] == df_film['film_id'])).\
                        join(df_rental, (df_rental['inventory_id'] == df_inventory['inventory_id'])).\
                        join(df_payment, (df_payment['rental_id'] == df_rental['rental_id']))
df_max_sum_amount = df_joined_three.groupBy(df_film_category['category_id']).\
                    agg(_sum(df_payment['amount']).alias('sum')).orderBy(col('sum').desc()).limit(1)

df_joined_category = df_max_sum_amount.join(df_category, (df_category['category_id'] == df_max_sum_amount['category_id']))
df_max_amount_category = df_joined_category.select([df_category['name'], "sum"])

In [26]:
df_max_amount_category.show()

+-------+--------+
|   name|     sum|
+-------+--------+
|Foreign|10507.67|
+-------+--------+



## Task 4

In [35]:
df_joined_four = df_film.join(df_inventory, 'film_id', 'left')
df_films = df_joined_four.where(df_inventory['film_id'].isNull()).select(['title'])

In [38]:
df_films.show()

+--------------------+
|               title|
+--------------------+
|      CHOCOLATE DUCK|
|       BUTCH PANTHER|
|        VOLUME HOUSE|
|      ORDER BETRAYED|
|        TADPOLE PARK|
|    KILL BROTHERHOOD|
|FRANKENSTEIN STRA...|
|    CROSSING DIVORCE|
|    SUICIDES SILENCE|
|       CATCH AMISTAD|
|     PERDITION FARGO|
|       FLOATS GARDEN|
|           GUMP DATE|
|        WALLS ARTIST|
|  GLADIATOR WESTWARD|
|         HOCUS FRIDA|
|ARSENIC INDEPENDENCE|
|         MUPPET MILE|
|   FIREHOUSE VIETNAM|
|       ROOF CHAMPION|
+--------------------+
only showing top 20 rows


## Task 5

In [74]:
df_cat_children = df_category.where(df_category['name'] == 'Children').select(['category_id'])
df_joined_five = df_film_actor.join(df_film, 'film_id').join(df_film_category, 'film_id').join(df_cat_children, 'category_id')
df_actors_grouped = df_joined_five.groupBy(df_film_actor['actor_id']).\
                    agg(count_distinct('title').alias('count'))
win = Window.orderBy(col('count').desc())
df_ranked = df_actors_grouped.withColumn("rank", dense_rank().over(win)).where(col('rank') < 4)
df_ranked_actors = df_ranked.join(df_actor, 'actor_id').\
    select(['actor_id','first_name','last_name', 'rank','count']).\
    orderBy(col('rank'), col('first_name'), col('last_name'))

In [75]:
df_ranked_actors.collect()

25/12/03 11:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/03 11:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/03 11:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/03 11:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/03 11:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/03 11:52:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/03 1

[Row(actor_id=139, first_name='EWAN', last_name='GOODING', rank=1, count=9),
 Row(actor_id=133, first_name='RICHARD', last_name='PENN', rank=1, count=9),
 Row(actor_id=105, first_name='SIDNEY', last_name='CROWE', rank=1, count=9),
 Row(actor_id=29, first_name='ALEC', last_name='WAYNE', rank=2, count=8),
 Row(actor_id=56, first_name='DAN', last_name='HARRIS', rank=2, count=8),
 Row(actor_id=142, first_name='JADA', last_name='RYDER', rank=2, count=8),
 Row(actor_id=131, first_name='JANE', last_name='JACKMAN', rank=2, count=8),
 Row(actor_id=145, first_name='KIM', last_name='ALLEN', rank=2, count=8),
 Row(actor_id=66, first_name='MARY', last_name='TANDY', rank=2, count=8),
 Row(actor_id=181, first_name='MATTHEW', last_name='CARREY', rank=2, count=8),
 Row(actor_id=149, first_name='RUSSELL', last_name='TEMPLE', rank=2, count=8),
 Row(actor_id=87, first_name='SPENCER', last_name='PECK', rank=2, count=8),
 Row(actor_id=65, first_name='ANGELA', last_name='HUDSON', rank=3, count=7),
 Row(actor

## Task 6

In [116]:
# select city.city, count(case when customer.active = 1 then 1 else 0 end)as active_customers, 
# count(case when customer.active = 0 then 1 else 0 end)as inactive_customers from customer 
# inner join address on customer.address_id=address.address_id 
# inner join city on address.city_id=city.city_id group by city.city order by inactive_customers desc, city.city;

In [114]:
df_joined_six = df_customer.join(df_address, 'address_id').join(df_city, 'city_id')
df_cities_grouped = df_joined_six.groupBy(df_city['city']).\
    agg(_sum(when(df_joined_six['active'] == 1, 1).otherwise(0)).alias('active_customers'),
       _sum(when(df_joined_six['active'] == 0, 1).otherwise(0)).alias('inactive_customers')).orderBy(col('inactive_customers').desc(), 'city')

In [115]:
df_cities_grouped.show()

+------------------+----------------+------------------+
|              city|active_customers|inactive_customers|
+------------------+----------------+------------------+
|            Amroha|               0|                 1|
|           Bat Yam|               0|                 1|
|  Charlotte Amalie|               0|                 1|
|     Coatzacoalcos|               0|                 1|
|            Daxian|               0|                 1|
|            Kamyin|               0|                 1|
|            Ktahya|               0|                 1|
|        Kumbakonam|               0|                 1|
|         Najafabad|               0|                 1|
|         Pingxiang|               0|                 1|
|   Southend-on-Sea|               0|                 1|
|       Szkesfehrvr|               0|                 1|
|          Uluberia|               0|                 1|
|           Wroclaw|               0|                 1|
|          Xiangfan|           

## Task 7

--7)select category.name, extract (hour from sum(rental.return_date - rental.rental_date)) as total_rent_hours from
--(select city_id, city from city where lower(city) like 'a%' or city like '%-%') as city_query inner join address on city_query.city_id=address.city_id
--inner join customer on address.address_id=customer.address_id
--inner join rental on customer.customer_id=rental.customer_id
--inner join inventory on rental.inventory_id=inventory.inventory_id
--inner join film_category on inventory.film_id=film_category.film_id
--inner join category on film_category.category_id=category.category_id group by category.name order by total_rent_hours desc limit 1;

In [176]:
df_city_filtered = df_city.where((lower(df_city['city']).like('a%')) | (df_city['city'].like('%-%')))

df_joined_seven = df_city_filtered.join(df_address, 'city_id').\
                                    join(df_customer, 'address_id').\
                                    join(df_rental, 'customer_id').\
                                    join(df_inventory, 'inventory_id').\
                                    join(df_film_category, 'film_id')
df_rental_hours_by_category = df_joined_seven.groupBy(df_film_category['category_id']).\
                            agg(
                                _sum(
                                    extract(lit('hours'), df_joined_seven['return_date'] - df_joined_seven['rental_date'])
                                ).alias('total_rent_hours')
                            ).orderBy(col('total_rent_hours'))
df_res_category = df_rental_hours_by_category.join(df_category, 'category_id').\
    orderBy(col('total_rent_hours').desc()).limit(1).select(['name', 'total_rent_hours'])

In [177]:
df_res_category.show()

+-----+----------------+
| name|total_rent_hours|
+-----+----------------+
|Drama|            3148|
+-----+----------------+

